# Notebook 2: Building the Bayesian Scoring Layer

Part of *Quantifying AI Risk*. Hour 2 of 3.

We just spent twenty minutes on why the scoring layer has to be Bayesian and not a weighted average. Now we build it. By the end of this notebook you will have a scoring engine that reads the telemetry stream from Hour 1, maintains a Beta distribution per governance pillar, updates it on every event using severity weights, aggregates the pillars into a system level score with a confidence interval, and produces a color coded output that a non technical executive can read at a glance.

Hour 3 reads from this. The Monte Carlo risk engine takes these scores and turns them into dollar denominated risk. So the scoring layer is the middle of the stack, and it is where the telemetry stream first becomes something a CFO can act on.

## Setup

Standard library plus one import from scipy for the Beta distribution itself. If scipy is unavailable, the math is simple enough that we could hand roll it. We use scipy because the confidence interval calculation is one line instead of ten.

In [ ]:
import random
import math
from collections import defaultdict
from scipy.stats import beta as beta_dist

random.seed(42)

print("Ready.")

## Load the telemetry stream

Notebook 1 wrote the stream to `data/telemetry_stream.jsonl`. We read from that file here, the same way a real production pipeline would. Hour 1 emits and persists. Hour 2 reads and scores. Hour 3 reads and simulates. Each stage writes to a known location so the next stage can pick up cleanly.

If the file does not exist (because Notebook 1 has not been run yet), we fall back to regenerating the stream in memory using the same seed and the same emit function. That keeps this notebook runnable on its own for students who want to skip ahead, while the production-style read-from-disk path is what we demonstrate first.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta

INPUT_PATH = Path("../data/telemetry_stream.jsonl") if Path("../data/telemetry_stream.jsonl").exists() else Path("data/telemetry_stream.jsonl")

if INPUT_PATH.exists():
    with open(INPUT_PATH) as f:
        stream = [json.loads(line) for line in f]
    print(f"Loaded {len(stream)} events from {INPUT_PATH.resolve()}")
else:
    # Fallback: regenerate the stream in memory if Notebook 1 has not been run.
    # Same seed, same emit function as Hour 1.
    print(f"No telemetry file found at {INPUT_PATH}. Regenerating in memory.")

    def emit_event_with_scenario(event_num):
        event = {
            "timestamp":    (datetime.now(timezone.utc) + timedelta(minutes=event_num)).isoformat(),
            "model_id":     "loan-approval-v2",
            "applicant_id": f"APP-{event_num:04d}",
            "decision":     random.choice(["APPROVED", "REJECTED"]),
        }
        if event_num < 600:
            event["confidence"]    = round(random.betavariate(8, 2), 3)
            event["latency_ms"]    = round(random.gauss(180, 35), 1)
            event["drift_score"]   = round(random.betavariate(2, 40), 4)
            event["fairness_flag"] = random.random() > 0.03
            event["memory_mb"]     = round(random.gauss(512, 30), 1)
        else:
            event["confidence"]    = round(random.betavariate(3, 3), 3)
            event["latency_ms"]    = round(random.gauss(320, 80), 1)
            event["drift_score"]   = round(random.betavariate(8, 10), 4)
            event["fairness_flag"] = random.random() > 0.15
            event["memory_mb"]     = round(random.gauss(620, 50), 1)
        return event

    random.seed(42)
    stream = [emit_event_with_scenario(i) for i in range(1000)]
    print(f"Regenerated {len(stream)} events.")

## Bring forward the governance map

Same signal to dimension map we built in Hour 1. This is what tells the scoring engine which signal feeds which pillar.

In [ ]:
GOVERNANCE_MAP = {
    "confidence": {
        "dimension": "Reliability & Robustness",
        "healthy":   0.70,
        "direction": "higher_is_better",
    },
    "latency_ms": {
        "dimension": "Operational Health",
        "healthy":   300,
        "direction": "lower_is_better",
    },
    "drift_score": {
        "dimension": "Data & Model Integrity",
        "healthy":   0.10,
        "direction": "lower_is_better",
    },
    "fairness_flag": {
        "dimension": "Fairness & Societal Impact",
        "healthy":   0.95,
        "direction": "higher_is_better",
    },
    "memory_mb": {
        "dimension": "Resilience & Continuity",
        "healthy":   600,
        "direction": "lower_is_better",
    },
}

print("Map loaded. 5 signals across 5 pillars.")

## The skeptical prior

A new system starts untrusted until it proves otherwise. That is the zero trust principle applied to AI, and it is the single most important design choice in this whole notebook.

In Beta distribution terms, the prior is two numbers: alpha and beta. Alpha counts evidence of trustworthy behavior, beta counts evidence of untrustworthy behavior. The mean of the distribution is alpha divided by alpha plus beta. A new pillar gets alpha=2 and beta=8, which gives a mean of 0.20. That is skeptical. It says "I do not trust this yet, and you are going to have to show me a lot of good evidence before I change my mind."

Compare that to the naive choice of alpha=1 and beta=1, which gives a uniform prior with mean 0.50. That is "I have no opinion." In governance, you should have an opinion. Your opinion is that untested systems are untrustworthy. Encode it.

In [ ]:
PRIOR_ALPHA = 2   # evidence of trustworthy behavior, starting small
PRIOR_BETA  = 8   # evidence of untrustworthy behavior, starting larger


def make_pillar_state():
    return {"alpha": PRIOR_ALPHA, "beta": PRIOR_BETA}


# One state per pillar
pillar_state = {dim: make_pillar_state() for dim in set(info["dimension"] for info in GOVERNANCE_MAP.values())}

print("Prior state, one per pillar:\n")
for dim, state in pillar_state.items():
    mean = state["alpha"] / (state["alpha"] + state["beta"])
    print(f"  {dim:32s}  alpha={state['alpha']}  beta={state['beta']}  mean={mean:.2f}")
print("\nEvery pillar starts skeptical. Mean trustworthiness of 0.20.")
print("Telemetry has to earn the score upward.")

## Severity weights

Not every governance failure is equal. A latency spike of fifty milliseconds and a fairness violation are both "signal outside threshold," but they are not the same severity of evidence against trustworthiness. The scoring has to reflect that.

We apply severity at the likelihood level. When a healthy signal arrives, it increments alpha by its severity weight. When an unhealthy signal arrives, it increments beta by its severity weight. Fairness and drift carry the heaviest weights because regulatory frameworks treat them as the most consequential. Latency and memory carry the lightest weights because they are operational signals with less direct fundamental rights impact.

These specific numbers are illustrative. In your own deployment, you would derive severity weights from your regulatory scope, your business domain, and your risk register. The pattern is what matters.

In [ ]:
SEVERITY_WEIGHTS = {
    "confidence":    1.0,   # reliability, moderate weight
    "latency_ms":    0.5,   # operational, lower weight
    "drift_score":   2.0,   # data integrity, high weight
    "fairness_flag": 3.0,   # fairness, highest weight
    "memory_mb":     0.5,   # operational, lower weight
}

print("Severity weights:\n")
for signal, weight in sorted(SEVERITY_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"  {signal:14s}  {weight}x")
print("\nFairness and drift carry more evidence per event than latency or memory.")

## The Bayesian update

Here is the whole engine in one function. For each event, we look at each signal. If the signal is healthy, alpha goes up by the severity weight. If it is unhealthy, beta goes up by the severity weight. That is it. That is the Beta-Binomial update.

Fairness works slightly differently because it is a rolling rate signal, not a per event signal. We track the running fairness rate over the last 50 events and classify that, rather than classifying each flag individually. This matches how we handled fairness in Hour 1.

In [ ]:
def is_healthy(signal_name, value, gov_map):
    info = gov_map[signal_name]
    if info["direction"] == "lower_is_better":
        return value <= info["healthy"]
    else:
        return value >= info["healthy"]


def update_pillar(state, healthy, severity):
    if healthy:
        state["alpha"] += severity
    else:
        state["beta"] += severity


# Process the stream event by event, updating pillar states as we go
fairness_window = []
WINDOW = 50

# Snapshot trajectory so we can plot score over time later
trajectory = []

for i, event in enumerate(stream):
    # Per event signals
    for signal_name in ["confidence", "latency_ms", "drift_score", "memory_mb"]:
        value = event[signal_name]
        dim = GOVERNANCE_MAP[signal_name]["dimension"]
        healthy = is_healthy(signal_name, value, GOVERNANCE_MAP)
        update_pillar(pillar_state[dim], healthy, SEVERITY_WEIGHTS[signal_name])

    # Rolling rate signal: fairness
    fairness_window.append(event["fairness_flag"])
    if len(fairness_window) > WINDOW:
        fairness_window.pop(0)

    if len(fairness_window) == WINDOW:
        rate = sum(fairness_window) / WINDOW
        dim = GOVERNANCE_MAP["fairness_flag"]["dimension"]
        healthy = rate >= GOVERNANCE_MAP["fairness_flag"]["healthy"]
        update_pillar(pillar_state[dim], healthy, SEVERITY_WEIGHTS["fairness_flag"])

    # Snapshot every 25 events
    if i % 25 == 0:
        snapshot = {"event": i}
        for dim, state in pillar_state.items():
            snapshot[dim] = state["alpha"] / (state["alpha"] + state["beta"])
        trajectory.append(snapshot)

print("Update complete. Final pillar state:\n")
for dim in sorted(pillar_state.keys()):
    state = pillar_state[dim]
    mean = state["alpha"] / (state["alpha"] + state["beta"])
    print(f"  {dim:32s}  alpha={state['alpha']:7.1f}  beta={state['beta']:6.1f}  mean={mean:.3f}")

## The confidence interval

A point score by itself is not enough. 0.73 with tight bounds and 0.73 with wide bounds are completely different situations, and if you cannot tell them apart, you are reporting opinions not evidence.

The Beta distribution gives us the interval for free. The 95% credible interval is the range between the 2.5th and 97.5th percentiles of the distribution. Tight interval means lots of evidence. Wide interval means the score is tentative.

In [ ]:
def pillar_score(state, ci=0.95):
    a, b = state["alpha"], state["beta"]
    mean = a / (a + b)
    lower = beta_dist.ppf((1 - ci) / 2, a, b)
    upper = beta_dist.ppf(1 - (1 - ci) / 2, a, b)
    n_observations = a + b - (PRIOR_ALPHA + PRIOR_BETA)
    return mean, lower, upper, n_observations


print("Pillar scores with 95% credible intervals:\n")
print(f"  {'Pillar':32s}  {'Score':>8s}  {'Lower':>8s}  {'Upper':>8s}  {'Evidence':>10s}")
print(f"  {'-'*32}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*10}")
for dim in sorted(pillar_state.keys()):
    mean, lower, upper, n = pillar_score(pillar_state[dim])
    print(f"  {dim:32s}  {mean:>8.3f}  {lower:>8.3f}  {upper:>8.3f}  {n:>10.0f}")

print("\nThe right way to report this to a regulator is not '0.73.'")
print("It is '0.73 with 95% CI of [0.68, 0.78] based on N observations.'")

## The system level score

Each pillar has spoken for itself. Now we combine them into one number that an executive can read. We aggregate by weighting each pillar's posterior by its regulatory severity. Fairness and data integrity contribute more to the system score than operational health does, for the same reason they carried higher severity weights at the event level.

Notice what we are NOT doing. We are not averaging the signals before the Bayesian update. We are not collapsing the pillars early. Each pillar updates independently, then the posteriors are combined. That way when the system score drops, we can always say which pillar drove the drop. A collapsed-early system gives you one number and no explanation. A collapsed-late system gives you one number AND the explanation.

In [ ]:
PILLAR_WEIGHTS = {
    "Fairness & Societal Impact":   3.0,
    "Data & Model Integrity":       2.0,
    "Reliability & Robustness":     1.0,
    "Operational Health":           0.5,
    "Resilience & Continuity":      0.5,
}


def system_score(pillar_state, pillar_weights):
    numerator_mean = 0
    numerator_lower = 0
    numerator_upper = 0
    total_weight = sum(pillar_weights.values())

    for dim, weight in pillar_weights.items():
        mean, lower, upper, _ = pillar_score(pillar_state[dim])
        numerator_mean += weight * mean
        numerator_lower += weight * lower
        numerator_upper += weight * upper

    return (
        numerator_mean / total_weight,
        numerator_lower / total_weight,
        numerator_upper / total_weight,
    )


sys_mean, sys_lower, sys_upper = system_score(pillar_state, PILLAR_WEIGHTS)

print(f"System trust score:  {sys_mean:.3f}")
print(f"95% CI:              [{sys_lower:.3f}, {sys_upper:.3f}]")
print(f"Width:               {sys_upper - sys_lower:.3f}")

## The executive view

A 0 to 1 score is fine for engineers. For executives, we map it onto bands. Green, amber, red. These bands correspond to the decision you want the reader to make.

Green means deploy and monitor. Amber means deployed but requires attention. Red means the system should not be in production in its current state. The specific band thresholds below are illustrative. Your organization will set its own based on regulatory scope and business risk tolerance.

In [ ]:
def band(score):
    if score >= 0.80: return "GREEN"
    if score >= 0.60: return "AMBER"
    return "RED"


def band_meaning(b):
    return {
        "GREEN": "Deploy and monitor.",
        "AMBER": "Deployed, requires attention.",
        "RED":   "Should not be in production in current state.",
    }[b]


print("Executive view\n")
print(f"  System:  {band(sys_mean):5s}  {sys_mean:.3f}  CI [{sys_lower:.3f}, {sys_upper:.3f}]")
print(f"           {band_meaning(band(sys_mean))}\n")

print("Pillar breakdown:")
for dim in sorted(pillar_state.keys()):
    mean, lower, upper, n = pillar_score(pillar_state[dim])
    print(f"  [{band(mean):5s}]  {dim:32s}  {mean:.3f}  CI [{lower:.3f}, {upper:.3f}]")

print("\nWhen the system score drops, the pillar breakdown tells you which")
print("dimension drove the drop. That is the audit trail a regulator expects.")

## Watching the posterior evolve

One last view. We snapshotted the pillar means every 25 events as we processed the stream. Plotting those snapshots shows how each pillar's posterior moved over time, and in particular how the drift injection at event 600 showed up in each pillar's trajectory.

If you do not have matplotlib available, the text output below still tells the story.

In [ ]:
# Text based trajectory so this runs without matplotlib
print("Pillar mean trajectory (every 25 events)\n")
print(f"  {'Event':>6s}  " + "  ".join(f"{d[:16]:>16s}" for d in sorted(pillar_state.keys())))
print(f"  {'-'*6}  " + "  ".join(f"{'-'*16}" for _ in pillar_state.keys()))

for snap in trajectory[::4]:  # every 100 events
    row = f"  {snap['event']:>6d}  "
    for dim in sorted(pillar_state.keys()):
        row += f"{snap[dim]:>16.3f}  "
    print(row)

print("\nNotice how every pillar climbs during the healthy phase (events 0-599)")
print("and either stalls or degrades once drift begins at event 600.")
print("Fairness and Data Integrity move most because their severity weights")
print("make them most responsive to the signal degradation.")

## One more thing: why not a weighted average?

Before we close, let's actually prove the thing we claimed in the theory section. We said a weighted average would be unable to distinguish a system with a long healthy history from a system that was always borderline. Let's show that with numbers.

Imagine two systems. System A has 500 healthy events followed by 50 unhealthy ones. System B has 50 healthy events followed by 500 unhealthy ones, then a recovery where the last 50 are healthy. Both systems have exactly the same final 50 events (healthy) and similar overall unhealthy event counts. A weighted moving average gives them similar scores. The Bayesian update does not.

In [ ]:
def simulate_weighted_avg(observations):
    # Simple exponentially weighted average of healthy rate
    alpha_ewma = 0.1
    rate = 0.5  # start neutral
    for obs in observations:
        rate = alpha_ewma * (1 if obs else 0) + (1 - alpha_ewma) * rate
    return rate


def simulate_bayesian(observations, severity=1.0):
    a, b = PRIOR_ALPHA, PRIOR_BETA
    for obs in observations:
        if obs:
            a += severity
        else:
            b += severity
    return a / (a + b)


# System A: long healthy history, recent stumble
system_a = [True] * 500 + [False] * 50

# System B: short healthy history, long stumble, recent recovery
system_b = [True] * 50 + [False] * 500 + [True] * 50

print("Comparing two systems with similar aggregate statistics\n")
print(f"  System A: 500 healthy, 50 unhealthy (recent stumble after long track record)")
print(f"  System B: 50 healthy, 500 unhealthy, 50 healthy (recovery attempt)\n")

print(f"  {'Method':32s}  {'System A':>10s}  {'System B':>10s}")
print(f"  {'-'*32}  {'-'*10}  {'-'*10}")
print(f"  {'Weighted moving average':32s}  {simulate_weighted_avg(system_a):>10.3f}  {simulate_weighted_avg(system_b):>10.3f}")
print(f"  {'Bayesian (Beta-Binomial)':32s}  {simulate_bayesian(system_a):>10.3f}  {simulate_bayesian(system_b):>10.3f}")

print("\nThe moving average converges toward the recent window regardless of history.")
print("The Bayesian score remembers. System A retains most of its earned trust")
print("despite the recent stumble. System B has to rebuild from accumulated doubt.")
print("That is the behavior we said we wanted.")

## Persist the pillar posteriors

Same discipline as Hour 1. Everything we just built lives in memory. Notebook 3 is going to sample from these posteriors thousands of times to run a Monte Carlo simulation. For that to work, the posteriors have to survive past this kernel.

We write the pillar state as JSON. One pillar per key. Each value carries the alpha and beta of the Beta posterior, plus the system-level weights so Notebook 3 has everything it needs to reconstruct the scoring without re-running this notebook.

In [ ]:
OUTPUT_DIR = Path("../data") if Path("../data").exists() else Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "pillar_posteriors.json"

payload = {
    "pillar_state":     pillar_state,
    "pillar_weights":   PILLAR_WEIGHTS,
    "prior_alpha":      PRIOR_ALPHA,
    "prior_beta":       PRIOR_BETA,
    "n_events":         len(stream),
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Wrote pillar posteriors to {OUTPUT_PATH.resolve()}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")
print()
print("Pillars persisted:")
for dim, state in pillar_state.items():
    print(f"  {dim:32s}  alpha={state['alpha']:.1f}  beta={state['beta']:.1f}")
print()
print("Notebook 3 will read from this same path.")

## Your turn: contradiction detection

In the theory section we said one of the strongest uses of this engine is catching the organization with perfect policies on paper and a biased system in production. The mechanism is two evidence streams feeding the same pillar.

Your task:

1. Add a document evidence stream alongside the telemetry stream. For each pillar, simulate a few document evidence events. Policy exists = healthy. Policy missing = unhealthy.
2. Update the same pillar posteriors with both streams. Document evidence updates with lower severity than telemetry (weight 0.3 feels right, document existence is weak evidence of behavior).
3. Build a function that detects contradictions. If document evidence is strongly positive (say alpha_doc / total_doc > 0.8) and telemetry evidence is strongly negative (alpha_tel / total_tel < 0.4), that pillar is a contradiction.
4. Report which pillars are contradictions.

No solution code. In production, the hard part is not the math. The hard part is deciding what counts as a document event versus a telemetry event, and what threshold defines a meaningful contradiction. Practice those judgment calls now.

In [ ]:
# Your implementation

def emit_document_event(pillar_name):
    # Simulate a document evidence event: policy check, audit finding, etc.
    pass


def update_with_both_streams(pillar_state, telemetry_stream, document_stream):
    # Update pillar state using both streams with different severity weights
    pass


def detect_contradictions(pillar_state_tel, pillar_state_doc, threshold=0.3):
    # Return list of pillars where doc evidence and telemetry evidence disagree
    pass


### Your contradiction report

*Write three to five sentences here. Which pillars would you flag as contradictions in your own organization? What would you do about one? This written artifact is the kind of thing that goes into a governance committee report.*

## What we just built

A scoring layer. Specifically:

- A skeptical prior that treats untested systems as untrustworthy by default
- Severity weights derived from regulatory framework priorities, applied at the likelihood level
- A Beta-Binomial update that runs per pillar, not per system
- Confidence intervals on every score because a point estimate is an opinion
- A weighted aggregation from pillar posteriors to system score, preserving explainability
- A color coded executive view that maps scores to deployment decisions
- Proof that Bayesian beats weighted average when system history matters

Hour 3 reads from this. The Monte Carlo engine takes these pillar posteriors and turns them into a probability distribution of financial outcomes. The trust score is what the engineering team cares about. The dollar number is what the board needs. Both are built on top of the telemetry layer, and this layer is the bridge between them.

On to Hour 3.